# Nhóm 16 — Huấn luyện và kiểm chứng AI4I
Notebook đồng hành với gói chương trình mới. Chạy từ trên xuống. Trên Colab, tải gói ZIP mới lên ở ô đầu. Không cần dùng mô hình của notebook cũ. Test đã từng được quan sát trong quá trình phát triển; lần chạy lại này là tái lập. Hai ô đầu cài đặt và huấn luyện cần thời gian.

In [ ]:
from pathlib import Path
import zipfile, sys, subprocess, os
try:
    from google.colab import files
    uploaded=files.upload()  # Chọn Nhom16_MachineHealth.zip
    zips=[n for n in uploaded if n.endswith('.zip')]
    assert len(zips)==1, 'Chọn đúng một gói ZIP chương trình'
    with zipfile.ZipFile(zips[0]) as z: z.extractall('nhom16_program')
    os.chdir('nhom16_program')
except ImportError:
    print('Chạy local: đặt notebook cùng thư mục app.py')
assert Path('train.py').exists()
subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt'],check=True)

In [ ]:
import pandas as pd
df=pd.read_csv('data/ai4i2020.csv')
print('Kích thước:',df.shape)
display(pd.DataFrame({'Kiểu':df.dtypes.astype(str),'Thiếu':df.isna().sum(),'Phân biệt':df.nunique()}))
print('Trùng toàn dòng:',df.duplicated().sum())
display(df['Machine failure'].value_counts().rename('Số mẫu'))

## Huấn luyện thực sự
Lệnh dưới gọi train.py, trong đó các mô hình được học bằng fit. Tiền xử lý nằm trong pipeline. Script chia phân tầng, so sánh validation, chọn ngưỡng rồi mới đánh giá test. Không thay đổi seed để tìm kết quả đẹp.

In [ ]:
from datetime import datetime
run_dir='run_'+datetime.now().strftime('%Y%m%d_%H%M%S')
subprocess.run([sys.executable,'train.py','--csv','data/ai4i2020.csv','--output',run_dir,'--advanced'],check=True)

In [ ]:
from IPython.display import display, Image
for name in ['validation_comparison.csv','calibration_comparison.csv','metrics.csv']:
    print(name);display(pd.read_csv(Path(run_dir)/'reports'/name))
for name in ['pr_curve.png','overfit.png','confusion_test.png']:
    display(Image(filename=str(Path(run_dir)/'reports'/name)))

## Kiểm chứng pipeline và xuất kết quả
Nhãn chỉ dùng đối chiếu, không đưa vào đầu vào dự đoán. Đóng gói đồng bộ mô hình, dữ liệu và báo cáo của lần chạy này.

In [ ]:
import joblib, numpy as np
from core import predict_batch
b=joblib.load(Path(run_dir)/'mo_hinh_demo.joblib')
t=pd.read_csv(Path(run_dir)/'data/test_goc.csv')
r=predict_batch(b,t,b['metadata'])
ref=pd.read_csv(Path(run_dir)/'reports/predictions_test.csv')
assert np.allclose(r.probability,ref.probability)
print('Web pipeline khớp xác suất đã lưu.');display(r.head())

In [ ]:
import shutil
package=Path(run_dir)/'web';package.mkdir()
for n in ['app.py','core.py','evaluation.py','train.py','README.md','HUONG_DAN_TRINH_BAY.md']:
    shutil.copy(n,package/n)
for n in ['mo_hinh_demo.joblib','metadata.json','requirements.txt']:
    shutil.copy(Path(run_dir)/n,package/n)
for n in ['reports','data']:
    shutil.copytree(Path(run_dir)/n,package/n)
shutil.copy('data/mau_nhap.csv',package/'data/mau_nhap.csv')
shutil.copy('data/ai4i2020.csv',package/'data/ai4i2020.csv')
shutil.copytree('.streamlit',package/'.streamlit')
zip_path=shutil.make_archive(str(Path(run_dir)/'Nhom16_Web_TaiLap'),'zip',package)
try: files.download(zip_path)
except NameError: print(zip_path)